In [348]:
import math, pandas as pd
from math import gcd
from collections import Counter
from scipy.stats import chi2

def primeFactors(n):
    factors = set()
    i=2
    while i * i <= n:
        if n % i == 0:
            while n % i == 0:
                n //= i
        i += 1
    if n > 1:
        factors.add(n)
    return factors

#Test Hull-Dobbell (Vérifie si une séquence pseudo aléatoire est de période maximum)
def isFullPeriod(a, c, m):
    rule1 = gcd(c,m) == 1

    primeM = primeFactors(m)
    rule2 = all((a-1)%p == 0 for p in primeM)

    rule3 = (m % 4 !=0) or ((a-1) % 4 == 0)
    return rule1 and rule2 and rule3

#Formule congruentiel linéaire mixte (Générer la suite pseudo aléatoire)
def xnCompute(a, c , m, x0):
    xn = [x0]
    x1 = ((a * x0) + c) % m
    xn.append(x1)
    for i in range(m-1):
        xn.append(((a * xn[i]) + c) % m)
    return xn

#Test des fréquence
def unCompute(a,c,m,x0):
    xn = xnCompute(a,c,m,x0)
    un = []
    for i in range(len(xn)):
        un.append(xn[i] / m)
    return un

#Fréquence cumulée
def ynCompute(a,c,m,x0):
    un = unCompute(a,c,m,x0)
    yn =[]
    for i in range(len(un)):
        yn.append(int(un[i]*10))
    return yn

#test de saut (Savoir l'espace entre chaque nombre demandé dans la suite)
def jumpTest(a,c,m,x0, studiedNb):
    yn = ynCompute(a,c,m,x0)
    jump = []
    try:
        iPosition = yn.index(studiedNb)
    except ValueError:
        return jump 
    print(iPosition)

    for i in range(iPosition + 1, len(yn)):
        if(yn[i] == studiedNb):
            jump.append(i - iPosition - 1)
            iPosition = i
    return jump

#Test de course (Comparé les nombre 2 a 2 si le premier est supérieur au 2ème = 1 et si 1er supérieur = 2)
def courseTest(a, c, m, x0, size):
    xn = xnCompute(a, c, m, x0)
    course = []
    for i in range(0, size*2, 2):
        if xn[i] > xn[i + 1]:
            course.append(2)
        else:
            course.append(1)
    return course

#Permet de séparer le jeu de données and x groupe d'une taille y
def separating(a, c, m, x0, size, nbGroup):
    yn = ynCompute(a, c, m, x0)
    separated = []
    
    for i in range(0, size*nbGroup, size):
        temp = []
        for i in range(size):
            temp.append(yn[i])
        separated.append(temp)
    return separated

#compte le nombre de chaque combinaison possible dans un test de poker
def pokerCount(a,c,m,x0):
    suite = xnCompute(a,c,m,x0)
    poker = []
    #Dans l'ordre(Poker, Carré, Full, Brelan, Deux pair, une pair, rien)
    pokerCounter = [0,0,0,0,0,0,0]
    for i in range(0,len(suite), 5):
        if i + 5 <= len(suite):
            group= []
            for j in range(5):
                group.append(suite[i+j])
            poker.append(pokerCheck(group))
    for i in range(len(poker)):
        if poker[i] == "Poker":
            pokerCounter[0]+=1
        elif poker[i] == "Carré":
            pokerCounter[1]+=1
        elif poker[i] == "Full":
            pokerCounter[2]+=1
        elif poker[i] == "Brelan":
            pokerCounter[3]+=1
        elif poker[i] == "Deux Pair":
            pokerCounter[4]+=1
        elif poker[i] == "Une Pair":
            pokerCounter[5]+=1
        else:
            pokerCounter[6]+=1
    return pokerCounter

#Poker (dans groupe de 5 vérifie si il y a soit une pair, soit deux pair, soit un brelan(3 les memes) soit un carré(4 les memes) soit un full (une pair + un brelan) soit un poker(5 les memes) soit rien)
def pokerCheck(group):
    count = Counter(group)
    value = count.values()
    if 5 in value:
        return "Poker"
    elif 4 in value:
        return "Carré"
    elif 3 in value and 2 in value:
        return "Full"
    elif 3 in value:
        return "Brelan"
    elif list(value).count(2) ==2:
        return "Deux Pair"
    elif 2 in value:
        return "Une Pair"
    else:
        return "Rien"

#Compte combien de 0 et de 1 sont trouvé dans le test de course
def counting(a, c, m, x0):
    course = courseTest(a, c, m, x0)
    number = []
    one = 0
    two = 0
    for i in range(len(course)):
        if (course[i]) == 1:
            one+=1
        else:
            two+=1
    number.append(one)
    number.append(two)
    return number

#Test du carré-unité (Prendre nombre 4 a 4 pour en faire un graphique)
def carreUnit(a,c,m,x0, size):
    yn = ynCompute(a,c,m,x0)
    carreUnit = []
    for i in range(0, size*4, 4):
        carreUnit.append((yn[i+2] - yn[i])**2 + (yn[i+3] - yn[i+1])**2)
    return carreUnit

#Permet de faire la loi de poisson
def poisson(number):
    un = []
    unCumulated = []
    test = 0
    size = 0
    while(test < 1):
        temp1 = round((math.e** (-number)) * (number**size) / math.factorial(size), 3)
        test += temp1
        un.append(temp1)
        unCumulated.append(round(test, 3))
        size +=1
    return un, unCumulated

#K = inverse d'un de modulo "m" (Permet de trouvé le k pour avoir le x0 a partir de x1)
def kCompute(x1, modulo, a, c):
    k=0
    while((x1 - c + k * modulo) % a != 0):
        k+=1
    return k

def grouping(xi,ri,pi,npi,x):
    i = 0
    while i < len(npi) - 1:
        if npi[i] < 5:
            xi[i] += " + " + xi[i+1]
            ri[i] += ri[i+1]
            pi[i] += pi[i+1]
            npi[i] += npi[i+1]
            x[i] += x[i+1]
            del xi[i+1], ri[i+1], pi[i+1], npi[i+1], x[i+1]
            if i > 0:
                i-=1
        else:
            i+=1
    if len(npi) == 1 and npi[0] <5:
        return False
    return xi, ri, pi, npi, x

def generation(a, c, m, x0):

    full_period = isFullPeriod(a, c, m) 
    suite = xnCompute(a, c, m, x0)

    return(full_period,suite)

def frequence(a,c,m,x0):
    print("Etape 1 :")
    print("H0 : chaque chiffre apparait avec la meme frequence")
    print("H1 : la distribution diffère de l'uniforme")

    print("\nEtape 2 :")
    alpha = 0.05
    print(alpha)

    print("\nEtape 3 :")
    yn = ynCompute(a, c, m, x0)  

    xi = [0,1,2,3,4,5,6,7,8,9]
    ri = list(counting(yn).values())
    pi = [1/len(xi) for i in xi]
    npi = [sum(ri)/len(xi) for i in xi]
    i = [(ri[j] - npi[j])**2 / npi[j] for j in range(len(ri))]
    
    df_tab = pd.DataFrame({'xi': xi, 'ri': ri, 'pi': pi, 'npi': npi, '(ri-npi)²/npi': i})

    total = pd.DataFrame({'xi': ['Total'], 'ri': [df_tab['ri'].sum()], 'pi': [df_tab['pi'].sum()], 'npi': [df_tab['npi'].sum()], '(ri-npi)²/npi': [df_tab['(ri-npi)²/npi'].sum()]})
    
    df_tab_total = pd.concat([df_tab, total])
    print(df_tab_total.to_string(index=False))

    print("\nEtape 4 :")
    if df_tab['npi'][0] >= 5 and sum(ri) > 50:
        print("La condition est respectée pas besoin de regrouper")
        nb_modalite = df_tab['ri'].sum()
    else :
        grouping(xi, ri, pi, npi, i)
        dfg_tab = pd.DataFrame({'xi': xi, 'ri': ri, 'pi' : pi, "npi" : npi, '(ri-npi)²/npi': i})
        print(dfg_tab.to_string(index=False))
        nb_modalite = dfg_tab['ri'].sum()
    print("\nEtape 5 :")
    degre_liberte = nb_modalite - 1
    x2_obs_total = df_tab['(ri-npi)²/npi'].sum()

    valeur_critique = chi2.ppf(1 - alpha, degre_liberte)

    print(f"{x2_obs_total:.2f} =< {valeur_critique:.2f}")

    print("\nEtape 6 :")
    if x2_obs_total <= valeur_critique:
        print("H0 est acceptée : chaque chiffre apparait avec la même fréquence")
    else:
        print("H1 est acceptée : la distribution diffère de l'uniforme")

def poker(a,c,m,x0):
    print("Etape 1 :")
    print("H0 : la distribution des combinaisons (paire, double paire, brelan, etc.) correspond aux probabilités théoriques.")
    print("H1 : la distribution diffère de celle attendue")

    print("\nEtape2 :")
    alpha = 0.05
    print(alpha)

    print("\nEtape 3 :")
    
    xi = ["Poker", "Carré", "Full", "Brelan", "Deux Pair", "Une Pair", "Rien"]
    ri = pokerCount(a,c,m,x0)
    pi = [1/(10**4), 450/(10**5), 900/(10**5), 7200/(10**5), 10800/(10**5), 50400/(10**5), 0.3024]
    npi = [sum(ri)*p for p in pi]
    i = [(ri[j] - npi[j])**2 / npi[j] for j in range(len(ri))]

    dp_tab = pd.DataFrame({'xi': xi, 'ri': ri, 'pi' : pi, "npi" : npi, '(ri-npi)²/npi': i})
    print(dp_tab.to_string(index=False))

    print("\nEtape 4 :")
    grouped = grouping(xi,ri,pi,npi,i)
    if(grouped):
        dpg_tab = pd.DataFrame({'xi': xi, 'ri': ri, 'pi' : pi, "npi" : npi, '(ri-npi)²/npi': i})
        print(dpg_tab.to_string(index=False))
        nb_modalite = dpg_tab['ri'].sum()
        degre_liberte = nb_modalite - 1
        x2_obs_total = dpg_tab['(ri-npi)²/npi'].sum()
        print("\nX2 = %f" % x2_obs_total)
    else:
        print("Toujours inférieur a 5 aprés regroupement de toutes les catégories")
        nb_modalite = dp_tab['ri'].sum()
        degre_liberte = nb_modalite - 1
        x2_obs_total = dp_tab['(ri-npi)²/npi'].sum()
        print("\nX2 = %f" % x2_obs_total)
    
    print("\nEtape 5 :")

    valeur_critique = chi2.ppf(1 - alpha, degre_liberte)
    print("\nv = %f" % valeur_critique)

    print("\nEtape 6 :")
    
    if x2_obs_total <= valeur_critique:
        print("H0 est acceptée : la distribution des combinaisons (paire, double paire, brelan, etc.) correspond aux probabilités théoriques.")
    else:
        print("H1 est acceptée : la distribution diffère de celle attendue")
        
def partie1(a,c,m,x0):
    full_period, suite = generation(a, c, m, x0)

    if not full_period:
        print("Le 3 hypothèses du théorème de Hull-Dobell ne sont pas respectées")
    else:
        print("Les 3 hypothèses du théorème de Hull-Dobell sont respectées")
        print("\nTest de fréquence en six étapes :\n")
        frequence(a,c,m,x0)
        print("\nTest de poker en six étapes :\n")
        poker(a,c,m,x0)


In [349]:
def partie2(a,c,m,x0):

    poisson_client = poisson(1.5)
    poisson_client_prio = poisson(0.7)
    prio_absolu = 0,3
    
loi_duree_service = pd.DataFrame({
        "Durée en minutes": [1, 2, 3, 4, 5, 6],
        "Répétition": [24, 18, 10, 3, 3, 2]
})

COUTS = {
    "presence_ord": 15 / 60,
    "presence_pr_rel": 35 / 60,
    "presence_pr_abs": 45 / 60,
    "occup_pr": 33 / 60,
    "occup_ord": 28 / 60,
    "inoccup": 18 / 60,
    "perte_pr": 20,
    "perte_ord": 15
}


def calcul_min_station():

        client = 1.5
        client_prio = 0.7

        total_activite = client + client_prio
        somme_répétition = loi_duree_service["Répétition"].sum()
        durée_moyenne_service = sum(loi_duree_service["Durée en minutes"] * loi_duree_service["Répétition"]) / somme_répétition

        calcul_psy = total_activite * durée_moyenne_service

        rounded_value = round(calcul_psy)
        print(rounded_value)




    

    

In [350]:
calcul_min_station()

5


In [351]:
import random
import math
from collections import deque, namedtuple

# ----------------------------
# Paramètres (modifiables)
# ----------------------------
LAMBDA_ORD = 1.5
LAMBDA_PRIO = 0.7
PRIO_ABSOLUTE_SHARE = 0.30  # 30% des prioritaires sont absolus

# Durées de service (minutes) : {durée: répétitions} => distribution empirique
SERVICE_REPETITIONS = {1: 24, 2: 18, 3: 10, 4: 3, 5: 3, 6: 2}

# Coûts par heure
COST_PRESENCE_ORD = 15.0
COST_PRESENCE_PRIO_REL = 35.0
COST_PRESENCE_PRIO_ABS = 45.0

COST_OCCUPATION_PRIO = 33.0
COST_OCCUPATION_ORD = 28.0
COST_INOCCUPATION = 18.0

COST_LOSS_PRIO = 20.0
COST_LOSS_ORD = 15.0

# Pertes: par défaut, pas de pertes (files infinies)
ENABLE_LOSSES = False
MAX_QUEUE_LEN_ABS = math.inf
MAX_QUEUE_LEN_REL = math.inf
MAX_QUEUE_LEN_ORD = math.inf

# Horizon de simulation pour les coûts globaux
SIM_MINUTES = 240  # 4 heures

# Nombre de stations testés (incluant le minimum = 5)
STATION_COUNTS = [5, 6, 7, 8]

# Seed optionnelle pour reproductibilité (décommenter si besoin)
# random.seed(42)

# ----------------------------
# Structures
# ----------------------------
Client = namedtuple("Client", ["type", "service_time"])  # type in {"ABS", "REL", "ORD"}

class Station:
    def __init__(self):
        self.client = None
        self.remaining = 0

    def is_free(self):
        return self.client is None

    def assign(self, client: Client):
        self.client = client
        self.remaining = client.service_time

    def tick(self):
        """Advance one minute of service time if occupied; return True if freed."""
        if self.client is not None:
            self.remaining -= 1
            if self.remaining <= 0:
                self.client = None
                self.remaining = 0
                return True  # freed
        return False

    def snapshot(self):
        if self.client is None:
            return {"state": "FREE", "type": None, "remaining": 0}
        return {"state": "BUSY", "type": self.client.type, "remaining": self.remaining}

# ----------------------------
# Outils de distribution
# ----------------------------
def sample_poisson(lmbda: float) -> int:
    # Knuth's algorithm for Poisson
    L = math.exp(-lmbda)
    k = 0
    p = 1.0
    while p > L:
        k += 1
        p *= random.random()
    return k - 1

def build_service_distribution(repetitions: dict):
    # Convert repetitions to cumulative distribution
    total = sum(repetitions.values())
    items = sorted(repetitions.items())
    cum = []
    acc = 0
    for duration, count in items:
        acc += count / total
        cum.append((duration, acc))
    return cum

def sample_service_time(cum_dist):
    u = random.random()
    for duration, c in cum_dist:
        if u <= c:
            return duration
    return cum_dist[-1][0]

SERVICE_CUM_DIST = build_service_distribution(SERVICE_REPETITIONS)

# ----------------------------
# Génération des arrivées
# ----------------------------
def generate_arrivals():
    n_ord = sample_poisson(LAMBDA_ORD)
    n_prio = sample_poisson(LAMBDA_PRIO)

    arrivals = []

    # Prioritaires: split into ABS vs REL
    n_abs = 0
    for _ in range(n_prio):
        if random.random() < PRIO_ABSOLUTE_SHARE:
            n_abs += 1
    n_rel = n_prio - n_abs

    for _ in range(n_abs):
        arrivals.append(Client("ABS", sample_service_time(SERVICE_CUM_DIST)))
    for _ in range(n_rel):
        arrivals.append(Client("REL", sample_service_time(SERVICE_CUM_DIST)))
    for _ in range(n_ord):
        arrivals.append(Client("ORD", sample_service_time(SERVICE_CUM_DIST)))

    # For logging convenience, return split lists too
    return arrivals, n_abs, n_rel, n_ord

# ----------------------------
# Placement selon les priorités
# ----------------------------
def fill_stations(stations, q_abs, q_rel, q_ord):
    # Priority order: ABS > REL > ORD
    for st in stations:
        if st.is_free():
            if q_abs:
                st.assign(q_abs.popleft())
            elif q_rel:
                st.assign(q_rel.popleft())
            elif q_ord:
                st.assign(q_ord.popleft())
            # else remains free

# ----------------------------
# Coûts par minute
# ----------------------------
def per_minute_costs(stations, q_abs, q_rel, q_ord):
    # Presence cost per minute
    # Presence includes clients in service + in queue
    count_abs = sum(1 for st in stations if st.client and st.client.type == "ABS") + len(q_abs)
    count_rel = sum(1 for st in stations if st.client and st.client.type == "REL") + len(q_rel)
    count_ord = sum(1 for st in stations if st.client and st.client.type == "ORD") + len(q_ord)

    presence_cost = (
        count_abs * (COST_PRESENCE_PRIO_ABS / 60.0) +
        count_rel * (COST_PRESENCE_PRIO_REL / 60.0) +
        count_ord * (COST_PRESENCE_ORD / 60.0)
    )

    # Station occupation/inoccupation per minute
    occ_prio = sum(1 for st in stations if st.client and st.client.type in {"ABS", "REL"})
    occ_ord = sum(1 for st in stations if st.client and st.client.type == "ORD")
    free = sum(1 for st in stations if st.is_free())

    occupation_cost = (
        occ_prio * (COST_OCCUPATION_PRIO / 60.0) +
        occ_ord * (COST_OCCUPATION_ORD / 60.0) +
        free * (COST_INOCCUPATION / 60.0)
    )

    return presence_cost + occupation_cost

# ----------------------------
# Pertes (optionnelles)
# ----------------------------
def apply_losses(q_abs, q_rel, q_ord):
    # If capacity limits are finite, drop excess and compute costs
    lost_prio = 0
    lost_ord = 0

    if len(q_abs) > MAX_QUEUE_LEN_ABS:
        over = len(q_abs) - MAX_QUEUE_LEN_ABS
        for _ in range(int(over)):
            q_abs.pop()  # drop last
        lost_prio += int(over)

    if len(q_rel) > MAX_QUEUE_LEN_REL:
        over = len(q_rel) - MAX_QUEUE_LEN_REL
        for _ in range(int(over)):
            q_rel.pop()
        lost_prio += int(over)

    if len(q_ord) > MAX_QUEUE_LEN_ORD:
        over = len(q_ord) - MAX_QUEUE_LEN_ORD
        for _ in range(int(over)):
            q_ord.pop()
        lost_ord += int(over)

    loss_cost = (lost_prio * COST_LOSS_PRIO) + (lost_ord * COST_LOSS_ORD)
    return loss_cost, lost_prio, lost_ord

# ----------------------------
# Journalisation minute
# ----------------------------
def snapshot_stations(stations):
    return [st.snapshot() for st in stations]

def snapshot_queue(q):
    return [{"type": c.type, "service": c.service_time} for c in list(q)]

# ----------------------------
# Simulation principale (une configuration de stations)
# ----------------------------
def simulate(station_count, minutes, detailed_first_20=False):
    stations = [Station() for _ in range(station_count)]
    q_abs = deque()
    q_rel = deque()
    q_ord = deque()

    total_cost = 0.0
    total_lost_prio = 0
    total_lost_ord = 0

    logs = []

    for t in range(1, minutes + 1):
        # 1) Décrément et libération
        for st in stations:
            st.tick()

        # 2) État début de minute (avant placement)
        if detailed_first_20 and t <= 20:
            log_entry = {
                "minute": t,
                "stations_begin": snapshot_stations(stations),
                "queues_begin": {
                    "ABS": snapshot_queue(q_abs),
                    "REL": snapshot_queue(q_rel),
                    "ORD": snapshot_queue(q_ord)
                },
                "arrivals": [],
                "queues_after_arrivals": None,
                "queues_after_placement": None,
                "stations_end": None
            }
        else:
            log_entry = None

        # 3) Générer arrivées et empiler
        arrivals, n_abs, n_rel, n_ord = generate_arrivals()
        # Empilement dans les files
        for c in arrivals:
            if c.type == "ABS":
                q_abs.append(c)
            elif c.type == "REL":
                q_rel.append(c)
            else:
                q_ord.append(c)

        if ENABLE_LOSSES:
            loss_cost, lost_prio, lost_ord = apply_losses(q_abs, q_rel, q_ord)
            total_cost += loss_cost
            total_lost_prio += lost_prio
            total_lost_ord += lost_ord

        if log_entry is not None:
            # Détail des arrivées (nombre et chaque client)
            log_entry["arrivals"] = [{"type": c.type, "service": c.service_time} for c in arrivals]
            log_entry["queues_after_arrivals"] = {
                "ABS": snapshot_queue(q_abs),
                "REL": snapshot_queue(q_rel),
                "ORD": snapshot_queue(q_ord)
            }

        # 4) Placement dans les stations libres (priorités)
        fill_stations(stations, q_abs, q_rel, q_ord)

        if log_entry is not None:
            log_entry["queues_after_placement"] = {
                "ABS": snapshot_queue(q_abs),
                "REL": snapshot_queue(q_rel),
                "ORD": snapshot_queue(q_ord)
            }

        # 5) Coûts de la minute
        total_cost += per_minute_costs(stations, q_abs, q_rel, q_ord)

        # 6) État fin de minute (après service d’une minute)
        if log_entry is not None:
            log_entry["stations_end"] = snapshot_stations(stations)
            logs.append(log_entry)

    results = {
        "station_count": station_count,
        "total_cost": total_cost,
        "total_lost_prio": total_lost_prio,
        "total_lost_ord": total_lost_ord,
        "logs": logs
    }
    return results

# ----------------------------
# Exécution: détails pour 5 stations, coûts pour autres
# ----------------------------
def main():
    all_results = []

    for s in STATION_COUNTS:
        detailed = (s == 5)
        minutes = SIM_MINUTES
        # Pour 5 stations: on veut les 20 premières minutes détaillées,
        # mais aussi les coûts en fin de simulation. Le code journalise les 20 premières.
        res = simulate(s, minutes, detailed_first_20=detailed)
        all_results.append(res)

    # Sorties: impression
    for res in all_results:
        s = res["station_count"]
        print(f"\n=== Résultats pour {s} stations ===")
        print(f"Coût total (sur {SIM_MINUTES} minutes): {res['total_cost']:.2f} €")
        if ENABLE_LOSSES:
            print(f"Pertes: prioritaires={res['total_lost_prio']}, ordinaires={res['total_lost_ord']}")
        else:
            print("Pertes: non activées (files infinies)")

        if s == 5:
            print("\n--- Détails minute par minute (20 premières minutes) ---")
            for log in res["logs"][:20]:
                t = log["minute"]
                print(f"\nMinute {t}:")
                print("Début de minute - Stations:")
                for i, st in enumerate(log["stations_begin"]):
                    print(f"  Station {i+1}: state={st['state']}, type={st['type']}, remaining={st['remaining']}")

                print("Début de minute - Files (avant placement):")
                for qname in ["ABS", "REL", "ORD"]:
                    q = log["queues_begin"][qname]
                    content = ", ".join([f"{item['type']}({item['service']})" for item in q]) or "vide"
                    print(f"  File {qname}: {content}")

                print("Arrivées:")
                if log["arrivals"]:
                    arr_txt = ", ".join([f"{a['type']}({a['service']})" for a in log["arrivals"]])
                    print(f"  {arr_txt}")
                else:
                    print("  Aucune")

                print("Après arrivées - Files:")
                for qname in ["ABS", "REL", "ORD"]:
                    q = log["queues_after_arrivals"][qname]
                    content = ", ".join([f"{item['type']}({item['service']})" for item in q]) or "vide"
                    print(f"  File {qname}: {content}")

                print("Après placement - Files:")
                for qname in ["ABS", "REL", "ORD"]:
                    q = log["queues_after_placement"][qname]
                    content = ", ".join([f"{item['type']}({item['service']})" for item in q]) or "vide"
                    print(f"  File {qname}: {content}")

                print("Fin de minute - Stations:")
                for i, st in enumerate(log["stations_end"]):
                    print(f"  Station {i+1}: state={st['state']}, type={st['type']}, remaining={st['remaining']}")

    # Détermination du nombre optimal (minimum de coût)
    best = min(all_results, key=lambda r: r["total_cost"])
    print(f"\n>>> Nombre optimal de stations (sur {SIM_MINUTES} minutes): {best['station_count']} "
          f"avec coût total = {best['total_cost']:.2f} €")

if __name__ == "__main__":
    main()



=== Résultats pour 5 stations ===
Coût total (sur 240 minutes): 1266.83 €
Pertes: non activées (files infinies)

--- Détails minute par minute (20 premières minutes) ---

Minute 1:
Début de minute - Stations:
  Station 1: state=FREE, type=None, remaining=0
  Station 2: state=FREE, type=None, remaining=0
  Station 3: state=FREE, type=None, remaining=0
  Station 4: state=FREE, type=None, remaining=0
  Station 5: state=FREE, type=None, remaining=0
Début de minute - Files (avant placement):
  File ABS: vide
  File REL: vide
  File ORD: vide
Arrivées:
  REL(1), ORD(2), ORD(1)
Après arrivées - Files:
  File ABS: vide
  File REL: REL(1)
  File ORD: ORD(2), ORD(1)
Après placement - Files:
  File ABS: vide
  File REL: vide
  File ORD: vide
Fin de minute - Stations:
  Station 1: state=BUSY, type=REL, remaining=1
  Station 2: state=BUSY, type=ORD, remaining=2
  Station 3: state=BUSY, type=ORD, remaining=1
  Station 4: state=FREE, type=None, remaining=0
  Station 5: state=FREE, type=None, remaini

In [ ]:
import math
import pandas as pd
from math import gcd
from collections import Counter
from scipy.stats import chi2

# ============================================================================
# PARTIE 1: GÉNÉRATEUR DE NOMBRES PSEUDO-ALÉATOIRES ET TESTS STATISTIQUES
# ============================================================================

def primeFactors(n):
    """Trouve les facteurs premiers de n"""
    factors = set()
    i = 2
    while i * i <= n:
        if n % i == 0:
            factors.add(i)
            while n % i == 0:
                n //= i
        i += 1
    if n > 1:
        factors.add(n)
    return factors

def isFullPeriod(a, c, m):
    """Test Hull-Dobbell pour vérifier si la période est maximale"""
    rule1 = gcd(c, m) == 1
    primeM = primeFactors(m)
    rule2 = all((a-1) % p == 0 for p in primeM)
    rule3 = (m % 4 != 0) or ((a-1) % 4 == 0)
    return rule1 and rule2 and rule3

def xnCompute(a, c, m, x0, n=None):
    """Génère la suite pseudo-aléatoire selon la formule X(n+1) = (a*X(n) + c) mod m"""
    if n is None:
        n = m
    xn = [x0]
    for i in range(n):
        xn.append((a * xn[-1] + c) % m)
    return xn[:n+1]

def unCompute(a, c, m, x0):
    """Normalise la suite entre 0 et 1"""
    xn = xnCompute(a, c, m, x0)
    return [x / m for x in xn]

def ynCompute(a, c, m, x0):
    """Convertit en chiffres de 0 à 9"""
    un = unCompute(a, c, m, x0)
    return [int(u * 10) if u < 1 else 9 for u in un]

def counting(yn):
    """Compte les occurrences de chaque chiffre"""
    counter = Counter(yn)
    return {i: counter.get(i, 0) for i in range(10)}

def pokerCheck(group):
    """Vérifie la combinaison poker d'un groupe de 5 éléments"""
    count = Counter(group)
    values = sorted(count.values(), reverse=True)
    
    if values == [5]:
        return "Poker"
    elif values == [4, 1]:
        return "Carré"
    elif values == [3, 2]:
        return "Full"
    elif values == [3, 1, 1]:
        return "Brelan"
    elif values == [2, 2, 1]:
        return "Deux Pair"
    elif values == [2, 1, 1, 1]:
        return "Une Pair"
    else:
        return "Rien"

def pokerCount(a, c, m, x0):
    """Compte les combinaisons poker dans la suite"""
    suite = xnCompute(a, c, m, x0)
    pokerCounter = {"Poker": 0, "Carré": 0, "Full": 0, "Brelan": 0, 
                    "Deux Pair": 0, "Une Pair": 0, "Rien": 0}
    
    for i in range(0, len(suite) - 4, 5):
        group = suite[i:i+5]
        result = pokerCheck(group)
        pokerCounter[result] += 1
    
    return pokerCounter

def grouping(xi, ri, pi, npi, x):
    """Regroupe les catégories avec npi < 5"""
    i = 0
    while i < len(npi) - 1:
        if npi[i] < 5:
            xi[i] = str(xi[i]) + " + " + str(xi[i+1])
            ri[i] += ri[i+1]
            pi[i] += pi[i+1]
            npi[i] += npi[i+1]
            x[i] += x[i+1]
            del xi[i+1], ri[i+1], pi[i+1], npi[i+1], x[i+1]
            if i > 0:
                i -= 1
        else:
            i += 1
    
    if len(npi) == 1 and npi[0] < 5:
        return False
    return True

def test_frequence(a, c, m, x0):
    """Test du chi-deux pour les fréquences"""
    print("\n" + "="*80)
    print("TEST DE FRÉQUENCE")
    print("="*80)
    
    print("\nÉtape 1 - Hypothèses :")
    print("H0 : Chaque chiffre apparaît avec la même fréquence (distribution uniforme)")
    print("H1 : La distribution diffère de l'uniforme")
    
    print("\nÉtape 2 - Seuil de signification :")
    alpha = 0.05
    print(f"α = {alpha}")
    
    print("\nÉtape 3 - Observations :")
    yn = ynCompute(a, c, m, x0)
    ri_dict = counting(yn)
    
    xi = list(range(10))
    ri = [ri_dict[i] for i in xi]
    pi = [1/10 for _ in xi]
    npi = [sum(ri)/10 for _ in xi]
    chi2_obs = [(ri[j] - npi[j])**2 / npi[j] for j in range(len(ri))]
    
    df_tab = pd.DataFrame({
        'xi': xi,
        'ri': ri,
        'pi': pi,
        'npi': npi,
        '(ri-npi)²/npi': chi2_obs
    })
    
    total = pd.DataFrame({
        'xi': ['Total'],
        'ri': [df_tab['ri'].sum()],
        'pi': [df_tab['pi'].sum()],
        'npi': [df_tab['npi'].sum()],
        '(ri-npi)²/npi': [df_tab['(ri-npi)²/npi'].sum()]
    })
    
    df_tab_total = pd.concat([df_tab, total], ignore_index=True)
    print(df_tab_total.to_string(index=False))
    
    print("\nÉtape 4 - Vérification des conditions :")
    if df_tab['npi'].min() >= 5 and sum(ri) > 50:
        print("✓ Toutes les fréquences théoriques ≥ 5 et n > 50")
        print("  Pas de regroupement nécessaire")
        degre_liberte = len(xi) - 1
        x2_obs_total = df_tab['(ri-npi)²/npi'].sum()
    else:
        print("✗ Regroupement nécessaire (npi < 5)")
        xi_copy = [str(x) for x in xi]
        ri_copy, pi_copy, npi_copy, chi2_copy = ri[:], pi[:], npi[:], chi2_obs[:]
        
        if grouping(xi_copy, ri_copy, pi_copy, npi_copy, chi2_copy):
            dfg_tab = pd.DataFrame({
                'xi': xi_copy,
                'ri': ri_copy,
                'pi': pi_copy,
                'npi': npi_copy,
                '(ri-npi)²/npi': chi2_copy
            })
            print("\nTableau après regroupement :")
            print(dfg_tab.to_string(index=False))
            degre_liberte = len(ri_copy) - 1
            x2_obs_total = sum(chi2_copy)
        else:
            print("Impossible de satisfaire la condition npi ≥ 5")
            degre_liberte = len(xi) - 1
            x2_obs_total = df_tab['(ri-npi)²/npi'].sum()
    
    print(f"\nÉtape 5 - Calcul de la statistique de test :")
    print(f"Degré de liberté : ν = {degre_liberte}")
    print(f"X² observé = {x2_obs_total:.4f}")
    
    valeur_critique = chi2.ppf(1 - alpha, degre_liberte)
    print(f"X² critique (α={alpha}, ν={degre_liberte}) = {valeur_critique:.4f}")
    
    print(f"\nÉtape 6 - Conclusion :")
    if x2_obs_total <= valeur_critique:
        print(f"✓ X² obs ({x2_obs_total:.4f}) ≤ X² crit ({valeur_critique:.4f})")
        print("  H0 acceptée : Distribution uniforme acceptable")
        return True
    else:
        print(f"✗ X² obs ({x2_obs_total:.4f}) > X² crit ({valeur_critique:.4f})")
        print("  H1 acceptée : Distribution non uniforme")
        return False

def test_poker(a, c, m, x0):
    """Test du chi-deux pour le poker"""
    print("\n" + "="*80)
    print("TEST DU POKER")
    print("="*80)
    
    print("\nÉtape 1 - Hypothèses :")
    print("H0 : Les combinaisons suivent les probabilités théoriques")
    print("H1 : La distribution diffère de celle attendue")
    
    print("\nÉtape 2 - Seuil de signification :")
    alpha = 0.05
    print(f"α = {alpha}")
    
    print("\nÉtape 3 - Observations :")
    poker_counts = pokerCount(a, c, m, x0)
    
    xi = ["Poker", "Carré", "Full", "Brelan", "Deux Pair", "Une Pair", "Rien"]
    ri = [poker_counts[x] for x in xi]
    pi = [0.0001, 0.0045, 0.0090, 0.0720, 0.1080, 0.5040, 0.3024]
    npi = [sum(ri) * p for p in pi]
    chi2_obs = [(ri[j] - npi[j])**2 / npi[j] for j in range(len(ri))]
    
    dp_tab = pd.DataFrame({
        'xi': xi,
        'ri': ri,
        'pi': pi,
        'npi': npi,
        '(ri-npi)²/npi': chi2_obs
    })
    print(dp_tab.to_string(index=False))
    
    print("\nÉtape 4 - Vérification des conditions :")
    xi_copy, ri_copy = xi[:], ri[:]
    pi_copy, npi_copy, chi2_copy = pi[:], npi[:], chi2_obs[:]
    
    if min(npi) < 5:
        print("✗ Regroupement nécessaire (npi < 5)")
        if grouping(xi_copy, ri_copy, pi_copy, npi_copy, chi2_copy):
            dpg_tab = pd.DataFrame({
                'xi': xi_copy,
                'ri': ri_copy,
                'pi': pi_copy,
                'npi': npi_copy,
                '(ri-npi)²/npi': chi2_copy
            })
            print("\nTableau après regroupement :")
            print(dpg_tab.to_string(index=False))
            degre_liberte = len(ri_copy) - 1
            x2_obs_total = sum(chi2_copy)
        else:
            print("Impossible de satisfaire npi ≥ 5")
            degre_liberte = len(xi) - 1
            x2_obs_total = sum(chi2_obs)
    else:
        print("✓ Toutes les fréquences théoriques ≥ 5")
        degre_liberte = len(xi) - 1
        x2_obs_total = sum(chi2_obs)
    
    print(f"\nÉtape 5 - Calcul de la statistique de test :")
    print(f"Degré de liberté : ν = {degre_liberte}")
    print(f"X² observé = {x2_obs_total:.4f}")
    
    valeur_critique = chi2.ppf(1 - alpha, degre_liberte)
    print(f"X² critique (α={alpha}, ν={degre_liberte}) = {valeur_critique:.4f}")
    
    print(f"\nÉtape 6 - Conclusion :")
    if x2_obs_total <= valeur_critique:
        print(f"✓ X² obs ({x2_obs_total:.4f}) ≤ X² crit ({valeur_critique:.4f})")
        print("  H0 acceptée : Distribution acceptable")
        return True
    else:
        print(f"✗ X² obs ({x2_obs_total:.4f}) > X² crit ({valeur_critique:.4f})")
        print("  H1 acceptée : Distribution non conforme")
        return False

# ============================================================================
# PARTIE 2: SYSTÈME D'ATTENTE AVEC PRIORITÉS
# ============================================================================

class Client:
    def __init__(self, id_client, type_client, temps_arrivee, duree_service):
        self.id = id_client
        self.type = type_client
        self.temps_arrivee = temps_arrivee
        self.duree_service = duree_service
        self.duree_restante = duree_service
        self.temps_debut_service = None
    
    def __repr__(self):
        return f"C{self.id}({self.type[0].upper()}{self.duree_service})"

class Station:
    def __init__(self, id_station):
        self.id = id_station
        self.client = None
    
    def est_libre(self):
        return self.client is None
    
    def affecter_client(self, client, temps_actuel):
        self.client = client
        client.temps_debut_service = temps_actuel
    
    def traiter_minute(self):
        if self.client:
            self.client.duree_restante -= 1
            if self.client.duree_restante <= 0:
                client_libere = self.client
                self.client = None
                return client_libere
        return None
    
    def __repr__(self):
        if self.client:
            return f"S{self.id}[{self.client}, reste={self.client.duree_restante}min]"
        return f"S{self.id}[Libre]"

class SystemeAttente:
    def __init__(self, nb_stations, generateur_rng, duree_simulation=200):
        self.nb_stations = nb_stations
        self.generateur = generateur_rng
        self.duree_simulation = duree_simulation
        
        # Paramètres
        self.lambda_ordinaire = 1.5
        self.lambda_prioritaire = 0.7
        self.proba_absolu = 0.3
        
        # Loi de service
        self.durees_service = [1, 2, 3, 4, 5, 6]
        self.repetitions = [24, 18, 10, 3, 3, 2]
        self.proba_cumul_service = []
        cumul = 0
        total = sum(self.repetitions)
        for r in self.repetitions:
            cumul += r / total
            self.proba_cumul_service.append(cumul)
        
        # Coûts (par minute)
        self.cout_presence_ordinaire = 15 / 60
        self.cout_presence_relatif = 35 / 60
        self.cout_presence_absolu = 45 / 60
        self.cout_occupation_ordinaire = 28 / 60
        self.cout_occupation_prioritaire = 33 / 60
        self.cout_inoccupation = 18 / 60
        
        # État
        self.stations = [Station(i+1) for i in range(nb_stations)]
        self.file_absolus = []
        self.file_relatifs = []
        self.file_ordinaires = []
        
        # Statistiques
        self.compteur_clients = 0
        self.cout_total_presence = 0
        self.cout_total_occupation = 0
        self.cout_total_inoccupation = 0
        self.clients_termines = []
    
    def generer_arrivee_poisson(self, lambda_param):
        """Génère inter-arrivée selon loi de Poisson"""
        u = self.generateur.get_uniform()
        return -math.log(1 - u) / lambda_param if u < 0.9999 else 10
    
    def generer_duree_service(self):
        """Génère durée de service selon distribution donnée"""
        u = self.generateur.get_uniform()
        for i, cumul in enumerate(self.proba_cumul_service):
            if u <= cumul:
                return self.durees_service[i]
        return self.durees_service[-1]
    
    def generer_clients_minute(self, minute):
        """Génère les clients arrivant durant cette minute"""
        clients = []
        
        # Clients ordinaires
        temps = self.generer_arrivee_poisson(self.lambda_ordinaire)
        if temps < 1:
            self.compteur_clients += 1
            duree = self.generer_duree_service()
            clients.append(Client(self.compteur_clients, 'ordinaire', minute, duree))
        
        # Clients prioritaires
        temps = self.generer_arrivee_poisson(self.lambda_prioritaire)
        if temps < 1:
            self.compteur_clients += 1
            duree = self.generer_duree_service()
            u = self.generateur.get_uniform()
            type_c = 'prioritaire_absolu' if u < self.proba_absolu else 'prioritaire_relatif'
            clients.append(Client(self.compteur_clients, type_c, minute, duree))
        
        return clients
    
    def afficher_etat(self, minute, phase):
        """Affiche l'état détaillé du système"""
        print(f"\n{'='*80}")
        print(f"Minute {minute} - {phase}")
        print(f"{'='*80}")
        
        print("\nStations:")
        for station in self.stations:
            print(f"  {station}")
        
        print(f"\nFiles d'attente:")
        print(f"  Absolus    ({len(self.file_absolus):2d}): {self.file_absolus}")
        print(f"  Relatifs   ({len(self.file_relatifs):2d}): {self.file_relatifs}")
        print(f"  Ordinaires ({len(self.file_ordinaires):2d}): {self.file_ordinaires}")
    
    def placer_clients(self, minute):
        """Place les clients en attente selon priorités"""
        files = [
            (self.file_absolus, 'absolu'),
            (self.file_relatifs, 'relatif'),
            (self.file_ordinaires, 'ordinaire')
        ]
        
        for file, _ in files:
            while file:
                station_libre = next((s for s in self.stations if s.est_libre()), None)
                if station_libre:
                    client = file.pop(0)
                    station_libre.affecter_client(client, minute)
                else:
                    break
    
    def calculer_couts_minute(self):
        """Calcule les coûts pour cette minute"""
        # Coût de présence dans les files
        self.cout_total_presence += len(self.file_absolus) * self.cout_presence_absolu
        self.cout_total_presence += len(self.file_relatifs) * self.cout_presence_relatif
        self.cout_total_presence += len(self.file_ordinaires) * self.cout_presence_ordinaire
        
        # Coût des stations
        for station in self.stations:
            if station.client:
                # Présence en service
                if station.client.type == 'prioritaire_absolu':
                    self.cout_total_presence += self.cout_presence_absolu
                    self.cout_total_occupation += self.cout_occupation_prioritaire
                elif station.client.type == 'prioritaire_relatif':
                    self.cout_total_presence += self.cout_presence_relatif
                    self.cout_total_occupation += self.cout_occupation_prioritaire
                else:
                    self.cout_total_presence += self.cout_presence_ordinaire
                    self.cout_total_occupation += self.cout_occupation_ordinaire
            else:
                self.cout_total_inoccupation += self.cout_inoccupation
    
    def simuler(self, afficher_details=False):
        """Exécute la simulation"""
        print(f"\n{'#'*80}")
        print(f"SIMULATION AVEC {self.nb_stations} STATIONS")
        print(f"{'#'*80}")
        
        for minute in range(1, self.duree_simulation + 1):
            if afficher_details and minute <= 20:
                self.afficher_etat(minute, "DÉBUT DE MINUTE")
            
            # Traiter les services en cours
            for station in self.stations:
                client_termine = station.traiter_minute()
                if client_termine:
                    self.clients_termines.append(client_termine)
            
            if afficher_details and minute <= 20:
                self.afficher_etat(minute, "AVANT PLACEMENT")
            
            # Générer arrivées
            nouveaux = self.generer_clients_minute(minute)
            
            if afficher_details and minute <= 20:
                print(f"\n{'*'*80}")
                print(f"ARRIVÉES minute {minute}:")
                if nouveaux:
                    for c in nouveaux:
                        print(f"  {c}")
                else:
                    print("  Aucune")
            
            # Ajouter aux files
            for client in nouveaux:
                if client.type == 'prioritaire_absolu':
                    self.file_absolus.append(client)
                elif client.type == 'prioritaire_relatif':
                    self.file_relatifs.append(client)
                else:
                    self.file_ordinaires.append(client)
            
            # Placer clients
            self.placer_clients(minute)
            
            if afficher_details and minute <= 20:
                self.afficher_etat(minute, "APRÈS PLACEMENT")
            
            # Calculer coûts
            self.calculer_couts_minute()
            
            if afficher_details and minute <= 20:
                self.afficher_etat(minute, "FIN DE MINUTE")
        
        self.afficher_resultats()
        return self.cout_total_presence + self.cout_total_occupation + self.cout_total_inoccupation
    
    def afficher_resultats(self):
        """Affiche résultats finaux"""
        cout_total = (self.cout_total_presence + self.cout_total_occupation + 
                     self.cout_total_inoccupation)
        
        print(f"\n{'='*80}")
        print(f"RÉSULTATS - {self.nb_stations} STATIONS")
        print(f"{'='*80}")
        print(f"Coût présence    : {self.cout_total_presence:.2f} €")
        print(f"Coût occupation  : {self.cout_total_occupation:.2f} €")
        print(f"Coût inoccupation: {self.cout_total_inoccupation:.2f} €")
        print(f"{'='*80}")
        print(f"COÛT TOTAL       : {cout_total:.2f} €")
        print(f"{'='*80}")
        print(f"Clients traités  : {len(self.clients_termines)}")
        print(f"En attente       : {len(self.file_absolus) + len(self.file_relatifs) + len(self.file_ordinaires)}")
        print(f"En service       : {sum(1 for s in self.stations if not s.est_libre())}")
        print(f"{'='*80}\n")

# ============================================================================
# GÉNÉRATEUR RNG BASÉ SUR SUITE PSEUDO-ALÉATOIRE
# ============================================================================

class GenerateurRNG:
    def __init__(self, a, c, m, x0):
        self.a = a
        self.c = c
        self.m = m
        self.x_current = x0
        self.index = 0
    
    def get_uniform(self):
        """Retourne nombre entre 0 et 1"""
        self.x_current = (self.a * self.x_current + self.c) % self.m
        self.index += 1
        return self.x_current / self.m

# ============================================================================
# PROGRAMME PRINCIPAL
# ============================================================================

def calculer_min_stations(lambda_ord, lambda_prio, durees, repetitions):
    """Calcule nombre minimum théorique de stations"""
    total_arrivees = lambda_ord + lambda_prio
    duree_moy = sum(d * r for d, r in zip(durees, repetitions)) / sum(repetitions)
    rho = total_arrivees * duree_moy
    return max(5, math.ceil(rho))

def main():
    print("\n" + "#"*80)
    print("PROJET DE SIMULATION - SYSTÈME D'ATTENTE AVEC PRIORITÉS")
    print("#"*80)
    
    # Paramètres générateur
    a, c, m, x0 = 21, 1, 100, 7
    
    print("\n" + "="*80)
    print("PARTIE 1: GÉNÉRATION ET VALIDATION DES NOMBRES PSEUDO-ALÉATOIRES")
    print("="*80)
    print(f"\nParamètres: a={a}, c={c}, m={m}, x0={x0}")
    
    # Vérification Hull-Dobbell
    print("\nVérification du théorème de Hull-Dobbell:")
    full_period = isFullPeriod(a, c, m)
    
    if full_period:
        print(f"✓ Les 3 conditions sont respectées")
        print(f"  Période maximale: {m}")
        
        # Tests statistiques
        test_freq_ok = test_frequence(a, c, m, x0)
        test_poker_ok = test_poker(a, c, m, x0)
        
        if test_freq_ok :
            print("\n""le test de fréquence est réussi")
        else :
            print("Le test de fréquence a échoué")

        if test_poker_ok :
            print("Le test du poker est réussi")
        else :
            print("Le test du poker a échoué")
    else:
        print("✗ Les conditions ne sont pas respectées")
        print("  Le générateur n'est pas acceptable")
        return
    
    # PARTIE 2: Simulation
    print("\n" + "="*80)
    print("PARTIE 2: SIMULATION DU SYSTÈME D'ATTENTE")
    print("="*80)
    
    # Calcul minimum théorique
    min_theo = calculer_min_stations(1.5, 0.7, [1,2,3,4,5,6], [24,18,10,3,3,2])
    print(f"\nNombre minimum théorique de stations: {min_theo}")
    print(f"Configurations testées: {max(5, min_theo)} à {max(5, min_theo) + 5}")
    
    # Test configurations
    nb_stations_test = list(range(max(5, min_theo), max(5, min_theo) + 6))
    resultats = {}
    
    for nb in nb_stations_test:
        gen = GenerateurRNG(a, c, m, x0)
        systeme = SystemeAttente(nb, gen, duree_simulation=200)
        afficher = (nb == nb_stations_test[0])
        cout = systeme.simuler(afficher_details=afficher)
        resultats[nb] = cout
    
    # Récapitulatif
    print(f"\n{'#'*80}")
    print("RÉCAPITULATIF COMPARATIF")
    print(f"{'#'*80}")
    print(f"{'Stations':<12} {'Coût Total (€)':<20}")
    print("-" * 32)
    for nb in sorted(resultats.keys()):
        marqueur = " ← OPTIMAL" if resultats[nb] == min(resultats.values()) else ""
        print(f"{nb:<12} {resultats[nb]:<20.2f}{marqueur}")
    
    nb_optimal = min(resultats, key=resultats.get)
    print(f"\n{'='*80}")
    print(f"CONFIGURATION OPTIMALE: {nb_optimal} stations")
    print(f"COÛT MINIMAL: {resultats[nb_optimal]:.2f} €")
    print(f"{'='*80}")

if __name__ == "__main__":
    main()


################################################################################
PROJET DE SIMULATION - SYSTÈME D'ATTENTE AVEC PRIORITÉS
################################################################################

PARTIE 1: GÉNÉRATION ET VALIDATION DES NOMBRES PSEUDO-ALÉATOIRES

Paramètres: a=21, c=1, m=100, x0=7

Vérification du théorème de Hull-Dobbell:
✓ Les 3 conditions sont respectées
  Période maximale: 100

TEST DE FRÉQUENCE

Étape 1 - Hypothèses :
H0 : Chaque chiffre apparaît avec la même fréquence (distribution uniforme)
H1 : La distribution diffère de l'uniforme

Étape 2 - Seuil de signification :
α = 0.05

Étape 3 - Observations :
   xi  ri  pi   npi  (ri-npi)²/npi
    0  11 0.1  10.1       0.080198
    1  10 0.1  10.1       0.000990
    2  10 0.1  10.1       0.000990
    3  10 0.1  10.1       0.000990
    4  10 0.1  10.1       0.000990
    5  10 0.1  10.1       0.000990
    6  10 0.1  10.1       0.000990
    7  10 0.1  10.1       0.000990
    8  10 0.1  10.1       0.0